# Tune reduced and full model

In [ ]:
import os
import sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH, SEED, LR_MODEL_BASE_DIR
from src.utils import ALPHA_LIST
import pandas as pd
import numpy as np

In [ ]:
SEG_TYPES = [
    "turns",
    "pt_noshort",
]
MODEL_NAMES = [
    "Llama-3.3-8B-Instruct",
    "Llama-3.3-70B-Instruct",
]
IMPACT_TYPES = ["self", "parent"]
PROB_AGG_TYPES = [
    "reduced",
    "block",
    "max",
    "avg",
    # "0_shot",
]
BASE_SAVE_DIR = BASE_PATH / "initial_preds"

## Get model preds at every:

- (Self, Parent) x Alpha Val X L1 Val x Repeat (1-10) x test fold combination for REDUCED
- (Self, Parent) x (block, avg, max) x (8B, 70B) x (turns, pt_noshort) x Alpha Val X L1 Val x Repeat (1-10) x test fold combination for FULL

In [ ]:
N_THREADS = 4
N_JOBS = 26
N_REPEATS = 10
N_SPLITS = 4
SPLIT_TOL = 0.2

In [ ]:
def create_cmd_str(
    impact_type,
    agg_type,
    alpha_val,
    df_path,
    base_save_dir,
    seed,
    n_repeats,
    split_tol,
    n_splits,
    n_jobs,
    max_iter=10000,
):
    """
    Create command for lr_preds

    TODO: Check if good file already exists for passed in configuration
    """
    cmd_str = f"export PYTHONPATH={BASE_PATH}; \
			export OMP_NUM_THREADS={N_THREADS}; \
			uv run python -m src.elastic_preds \
			--impact_type {impact_type} \
			--agg_type {agg_type}\
			--alpha_val {alpha_val}\
			--df_path {df_path}\
			--base_save_dir {base_save_dir}\
			--seed {seed}\
			--n_repeats {n_repeats}\
			--split_tol {split_tol}\
			--n_splits {n_splits}\
			--n_jobs {n_jobs}\
			--max_iter {max_iter}"
    return " ".join(cmd_str.split())


def make_reduced(impact_type, alpha_val):
    df_path = BASE_PATH / "data" / "cleaned" / "processed" / f"{impact_type}.parquet"
    save_dir = BASE_SAVE_DIR / "reduced" / impact_type / f"alpha_{alpha_val}"
    cmd_str = create_cmd_str(
        impact_type=impact_type,
        alpha_val=alpha_val,
        df_path=df_path,
        base_save_dir=save_dir,
        seed=SEED,
        n_repeats=N_REPEATS,
        split_tol=SPLIT_TOL,
        n_splits=N_SPLITS,
        n_jobs=N_JOBS,
        agg_type="reduced",
    )
    return cmd_str


def make_full_lr(seg_types, model_name, impact_type, alpha_val, agg_type):
    cmd_list = []
    for seg_type in seg_types:
        df_path = (
            BASE_PATH
            / "data"
            / "w_probs"
            / model_name
            / seg_type
            / f"{impact_type}.parquet"
        )
        save_dir = (
            BASE_SAVE_DIR
            / agg_type
            / impact_type
            / model_name
            / seg_type
            / f"alpha_{alpha_val}"
        )
        cmd_str = create_cmd_str(
            impact_type=impact_type,
            alpha_val=alpha_val,
            df_path=df_path,
            base_save_dir=save_dir,
            seed=SEED,
            n_repeats=N_REPEATS,
            split_tol=SPLIT_TOL,
            n_splits=N_SPLITS,
            n_jobs=N_JOBS,
            max_iter=50000,  # need more iters to converge bc of higher dimension w/ embeddings
            agg_type=agg_type,
        )
        cmd_list.append(cmd_str)
    return cmd_list


def make_full_shot(
    model_name,
    impact_type,
    alpha_val,
    temp=0.6,
    pred_type="proba-limited",
    desc_type="augment",
):
    df_path = (
        BASE_PATH
        / "data"
        / "w_0_shot_probs"
        / f"temp_{temp}"
        / pred_type
        / model_name
        / desc_type
        / f"{impact_type}.parquet"
    )
    save_dir = (
        BASE_SAVE_DIR
        / "0_shot"
        / impact_type
        / model_name
        / f"output_{pred_type}"
        / f"desc_{desc_type}"
        / f"alpha_{alpha_val}"
    )
    cmd_str = create_cmd_str(
        impact_type=impact_type,
        alpha_val=alpha_val,
        df_path=df_path,
        base_save_dir=save_dir,
        seed=SEED,
        n_repeats=N_REPEATS,
        split_tol=SPLIT_TOL,
        n_splits=N_SPLITS,
        n_jobs=N_JOBS,
        max_iter=50000,  # need more iters to converge bc of higher dimension w/ embeddings
        agg_type="0_shot",
    )
    return cmd_str

In [ ]:
CMD_DIR = BASE_PATH / "swarm_dir" / "commands"
full_cmd_list = []
shot_pred_types = [
    "proba-limited",
    # "proba",
    # "binary",
]
shot_desc_types = [
    "original",
    "augment",
]
for agg_type in PROB_AGG_TYPES:
    for impact_type in IMPACT_TYPES:
        swarm_path = CMD_DIR / agg_type / f"{impact_type}.swarm"
        swarm_path.parent.mkdir(exist_ok=True, parents=True)
        if swarm_path.exists():
            swarm_path.unlink()
        cmd_list = []
        for alpha_val in ALPHA_LIST:
            ## Reduced model
            if agg_type == "reduced":
                cmd_str = make_reduced(
                    impact_type=impact_type,
                    alpha_val=alpha_val,
                )
                cmd_list.append(cmd_str)
                continue
            ## Full model (block, max, avg)
            for model_name in MODEL_NAMES:
                if agg_type in ["block", "max", "avg"]:
                    lr_cmd_list = make_full_lr(
                        seg_types=SEG_TYPES,
                        model_name=model_name,
                        impact_type=impact_type,
                        alpha_val=alpha_val,
                        agg_type=agg_type,
                    )
                    cmd_list += lr_cmd_list
                ## 0 shot
                else:
                    for pred_type in shot_pred_types:
                        for desc_type in shot_desc_types:
                            cmd_str = make_full_shot(
                                model_name=model_name,
                                impact_type=impact_type,
                                alpha_val=alpha_val,
                                temp=0.6,
                                pred_type=pred_type,
                                desc_type=desc_type,
                            )
                            cmd_list.append(cmd_str)
        swarm_path.write_text("\n".join(cmd_list))
        full_cmd_list += cmd_list
print(f"Num commands: {len(full_cmd_list)}")

In [ ]:
import subprocess

BASE_SAVE_DIR.mkdir(exist_ok=True, parents=True)
for iteration, cmd in enumerate(full_cmd_list):
    print(f"{iteration +1}/{len(full_cmd_list)}...")
    subprocess.run(cmd, shell=True, check=True)

### Write to swarm

In [ ]:
from src.swarm import run_swarms

run_swarms(
    agg_list=PROB_AGG_TYPES,
    impact_list=["parent", "self"],
    swarm_log_dir=BASE_PATH / "swarm" / "logs",
    cmd_dir=CMD_DIR,
    n_jobs=N_JOBS,
    n_threads=N_THREADS,
)

## Get Best Params (via CV)
Optimize for one of:
- r2
- mse

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score
from tqdm.auto import tqdm

# OPTIMIZE = "r2"
OPTIMIZE = "mse"
print(f"Optimizng for: {OPTIMIZE}")

In [ ]:
def get_best_params_score(import_dir, show_progress=False, optimize="mse"):
    if optimize not in ["mse", "r2"]:
        raise ValueError(f"optimize param must be one of ['mse', 'r2'], got {optimize}")
    best_params_score_dict = {
        "avg_mse": 9999999,
        "stdev_mse": 9999999,
        "avg_r2": 0,
        "stdev_r2": 9999999,
        "l1": "None",
        "alpha": "None",
    }
    total_files = sum(
        1
        for alpha_dir in import_dir.iterdir()
        for l1_dir in alpha_dir.iterdir()
        for repeat_dir in l1_dir.iterdir()
        for _ in repeat_dir.iterdir()
    )
    with tqdm(
        total=total_files,
        desc="Scoring params",
        leave=False,  # disappears when done
        dynamic_ncols=True,
        disable=not show_progress,
    ) as pbar:
        for alpha_dir in import_dir.iterdir():
            alpha_val = alpha_dir.name.split("_")[1]
            for l1_dir in alpha_dir.iterdir():
                l1_val = l1_dir.name.split("_")[1]
                mse_list = []
                r2_list = []
                for repeat_dir in l1_dir.iterdir():
                    for split_file in repeat_dir.iterdir():
                        df_import = pd.read_parquet(split_file)
                        mse = mean_squared_error(
                            y_true=df_import["true"], y_pred=df_import["pred"]
                        )
                        mse_list.append(mse)
                        r2 = r2_score(
                            y_true=df_import["true"], y_pred=df_import["pred"]
                        )
                        r2_list.append(r2)
                        pbar.update(1)
                # get avg mse across folds/repeats for l1-alpha combo
                avg_mse = np.mean(mse_list)
                avg_r2 = np.mean(r2_list)
                mse_condition = (optimize == "mse") and (
                    avg_mse < best_params_score_dict["avg_mse"]
                )
                r2_condition = (optimize == "r2") and (
                    avg_r2 > best_params_score_dict["avg_r2"]
                )

                if mse_condition or r2_condition:
                    stdev_mse = np.std(mse_list)
                    stdev_r2 = np.std(r2_list)
                    best_params_score_dict = {
                        "avg_mse": round(avg_mse, 3),
                        "stdev_mse": round(stdev_mse, 3),
                        "avg_r2": round(avg_r2, 4),
                        "stdev_r2": round(stdev_r2, 4),
                        "l1": l1_val,
                        "alpha": alpha_val,
                    }
    return best_params_score_dict

### Reduced

In [ ]:
df_rows = []
for impact_dir in (BASE_SAVE_DIR / "reduced").iterdir():
    impact_name = impact_dir.name
    print(f"{impact_name}...")
    cur_config = {"impact_type": impact_name, "agg_type": "reduced"}
    best_params_score_dict = get_best_params_score(
        impact_dir,
        show_progress=True,
        optimize=OPTIMIZE,
    )
    new_row = {
        **cur_config,
        **best_params_score_dict,
    }
    df_rows.append(new_row)
reduced_result_df = pd.DataFrame(df_rows)
display(reduced_result_df)

### Full

In [ ]:
df_rows = []
for agg_dir in BASE_SAVE_DIR.iterdir():
    agg_type = agg_dir.name
    if agg_type == "reduced" or agg_dir.is_file():
        continue
    for impact_dir in agg_dir.iterdir():
        impact_name = impact_dir.name
        print(f"{agg_type}-{impact_name}...")
        for model_dir in impact_dir.iterdir():
            model_name = model_dir.name
            print(f"\t{model_name}...")
            cur_config = {
                "agg_type": agg_type,
                "impact_type": impact_name,
                "model_name": model_name,
            }
            if agg_type == "0_shot":
                for output_dir in model_dir.iterdir():
                    output_type = output_dir.name.split("_")[1]
                    for desc_dir in output_dir.iterdir():
                        desc_type = desc_dir.name.split("_")[1]
                        cur_config.update(
                            {
                                "seg_type": "-----",
                                "output_type": output_type,
                                "desc_type": desc_type,
                            }
                        )
                        best_params_score_dict = get_best_params_score(
                            desc_dir,
                            show_progress=True,
                            optimize=OPTIMIZE,
                        )
                        new_row = {
                            **cur_config,
                            **best_params_score_dict,
                        }
                        df_rows.append(new_row)
            else:
                for seg_dir in model_dir.iterdir():
                    seg_type = seg_dir.name
                    print(f"\t\t{seg_type}...")
                    cur_config.update(
                        {
                            "seg_type": seg_type,
                            "output_type": "-----",
                            "desc_type": "-----",
                        }
                    )
                    cur_config["seg_type"] = seg_type
                    best_params_score_dict = get_best_params_score(
                        seg_dir,
                        show_progress=True,
                        optimize=OPTIMIZE,
                    )
                    new_row = {
                        **cur_config,
                        **best_params_score_dict,
                    }
                    df_rows.append(new_row)
full_result_df = pd.DataFrame(df_rows)
display(full_result_df)

Format table

In [ ]:
other = "r2" if OPTIMIZE == "mse" else "mse"
ascend_list = (
    [True, True, True, False] if OPTIMIZE == "mse" else [True, True, False, True]
)
reorder_cols = [
    "agg_type",
    "impact_type",
    f"avg_{OPTIMIZE}",
    f"stdev_{OPTIMIZE}",
    f"avg_{other}",
    f"stdev_{other}",
    "l1",
    "alpha",
    "model_name",
    "seg_type",
    "desc_type",
    "output_type",
]
full_result_df_ordered = full_result_df[reorder_cols]
full_result_df_ordered.sort_values(
    by=[
        "agg_type",
        "impact_type",
        f"avg_{OPTIMIZE}",
        f"avg_{other}",
    ],
    ascending=ascend_list,
    na_position="last",
    inplace=True,
)
full_result_df_ordered

Export tables

In [ ]:
export_dir = BASE_PATH / "initial_results" / f"params_optimize_{OPTIMIZE}"
print(f"Exporting to: {export_dir}")
## Reduced
reduced_path = export_dir / "reduced.tsv"
if reduced_path.exists():
    reduced_path.unlink()
reduced_path.parent.mkdir(exist_ok=True, parents=True)
reduced_result_df.to_csv(reduced_path, sep="\t")
## Full
# full_path = export_dir / "full.tsv"
# if full_path.exists():
#     full_path.unlink()
# full_path.parent.mkdir(exist_ok=True, parents=True)
# full_result_df_ordered.to_csv(full_path, sep="\t")

# Experiment

- Check to see if LR model provides good output

In [ ]:
import pandas as pd

pd.set_option("display.max_columns", 500)

r = pd.read_parquet(
    "/home/mataray/DAWBA/data/w_probs/pt_noshort/Llama-3.3-70B-Instruct/self.parquet"
)
r.columns

In [ ]:
temp = r[~r["p_dep_25h_text_ep_high"].str.contains("Not applicable")][
    ["open_text_block", "prob_vector_block", "impact_score_parent"]
]
for idx, row in enumerate(temp.itertuples()):
    if idx in [0, 1, 2, 3]:
        continue
    print(row.open_text_block)
    print(row.prob_vector_block)
    print(row.impact_score_parent)
    break

In [ ]:
lr_model_dir = LR_MODEL_BASE_DIR / "turns" / "Llama-3.3-70B-Instruct"
lr_model_dict = {}
for outcome_file in lr_model_dir.iterdir():
    outcome_name = outcome_file.stem
    # Skips `any` and any comprehensive `a` or `c` outcomes
    if len(outcome_name) != 2:
        continue
    lr_model_dict[outcome_name] = 5
outcome_order = sorted(lr_model_dict.keys())
if len(outcome_order) != 28:
    raise ValueError(
        f"outcome_order should be length 28, got {len(outcome_order)}. \nOutcomes: {outcome_order}"
    )